### Username: aancha.14.2025
### Course: MSc Data Science (EMATM0048); Software Development: Programming and Algorithms (SDPA)

### Name: Aanchala Bhongade; Student no.: 2830709

### Part 2: Data Analysis

# Analysis of Bloackchain Security: Market Volatility vs. Crime Severity

## 1. Introduction
In this project, I investigate the relationship between cryptocurrency market conditions and security breaches. Specifically, does the 'hype' of Bull Market lead to more frequent or more severe financial crimes compared to a Bear Market?

## 1.1 Libraries
I am using standard Python Libraries for data collection and the analysis:
1. **'requests'**: To crawl data from external APIs (DefiLlama & CryptoCompare).
2. **'pandas'**: For data cleaning and merging.
4. **'matplotlib & seaborn'**: For data visualisation of distribution of financial losses.
5. **'scipy.status'**: Used for statistical analysis like hypothesis testing (T-Test).

In [66]:
import requests
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
from scipy import stats
from datetime import datetime

#setting the visual style
sns.set_theme(style="whitegrid")
print("Libraries are imported successfully....")

Libraries are imported successfully....


## 2. Data Collection

### 2.1 Source A: Crime Data (DefiLlama)
I am extracting the data from the **DefiLlama Hacks API**. It is a open souce dataset that tracks Decentralised Finance (DeFi) protocol breaches.
* **Method:** API Request ('requests.get')
* **Variables:** 'timestamp_s', 'target', 'amount_usd', 'category' and 'blockchain'.

In [67]:
#Getting hacks data from DefiLlama
url_hacksdata = "https://api.llama.fi/hacks"

try:
    print("Requesting the data..")
    response = requests.get(url_hacksdata)
    response.raise_for_status()
    data = response.json()
    
    #creating a dataframe with the help of list of dictionaries
    df_hacks = pd.DataFrame(data)
    
    #Renaming the columns head
    df_hacks = df_hacks.rename(columns={
        'date': 'timestamp_s',
        'name': 'target',
        'amount': 'amount_usd',
        'classification': 'category',
        'chain': 'blockchain'
    })
    
    df_hacks = df_hacks[['timestamp_s', 'target', 'amount_usd', 'category', 'blockchain']]
    
    
    print(f"Sucessfully collected {len(df_hacks)} hacks record.")
    
except Exception as e:
    print(f"Error in fetching the data: {e}")
    
df_hacks.head()

Requesting the data..
Sucessfully collected 436 hacks record.


,timestamp_s,target,amount_usd,category,blockchain
0,1711065600,Super Sushi Samurai,4800000.0,Protocol Logic,[Blast]
1,1709856000,Unizen,2100000.0,Protocol Logic,[Ethereum]
2,1621814400,AutoShark,745000.0,Ecosystem,[BSC]
3,1630627200,Siren,3450000.0,Smart Contract Language,[Polygon]
4,1704585600,MangoFarmSol,2000000.0,Rugpull,[Solana]


### 2.2 Source B: Market Price Data (CryptoCompare)
I am collecting this data for comparing hacks against market price fluctuations, getting the hostorical Ethereum prices.
* **Method:** using *CryptoCompare* for data, it allows to get hacks for approx. 2000 days.
* **Variables:** 'timestamp_ms' and 'eth price'.

In [77]:
#collecting market price data from Binanace Public API

url_market = "https://min-api.cryptocompare.com/data/v2/histoday"
params = {'fsym': 'ETH', 'tysm': 'USD', 'limit': '2000'}

try:
    print("Requesting the dataset..")
    cc_response = requests.get(url_market, params=params)
    cc_data = cc_response.json()
    
    #fetching and renaming the columns of interest
    price_list = cc_data.get('Data', {}).get('Data', [])
    df_market = pd.DataFrame(price_list)
    df_market = df_market[['time', 'close']].rename(columns={
        'time': 'timestamp',
        'close': 'eth_price'
    })
    
    print(f"Successfully collected {len(df_market)} days of price data.")
    
    
    
except Exception as e:
    print(f"error in fetching price data: {e}")

print(df_market.head())

Requesting the dataset..
error in fetching price data: "None of [Index(['time', 'close'], dtype='object')] are in the [columns]"
Empty DataFrame
Columns: []
Index: []


## 3. Data Preparation and Cleaning
The two datasets uses different time formats, DefiLlama uses unix seconds, while CoinGecko uses unix miliseconds.

**Steps involved:**
1. **Data normalisation:** converting both time formats to python 'datetime' and extracting just the 'date' (YYYY-MM-DD).
2. **Filtering:** removal of small hacks (under $100k).
3. **Merging:** merging inner join on 'date' column to link every hack to the ETH price on that particular date.

In [65]:
#converting DefiLlama dates (unix seconds)
df_hacks['date_obj'] = pd.to_datetime(df_hacks['timestamp_s'], unit='s')
df_hacks['date_only'] = df_hacks['date_obj'].dt.date

#converting CoinGecko dates (unix miliseconds)
df_market['date_obj'] = pd.to_datetime(df_market['timestamp_ms'], unit='ms')
df_market['date_only'] = df_market['date_obj'].dt.date

#merging the datasets
df_final = pd.merge(df_hacks, df_market, on='date_only', how='inner')


print(f"Dataset is merged with the dimensions as following: {df_final.shape}")
df_final.head()

Dataset is merged with the dimensions as following: (91, 18)


,timestamp_s,target,category,technique,amount (usd),blockchain,bridgeHack,targetType,source,returnedFunds,defillamaId,language,parentProtocolId,date_obj_x,date_only,timestamp_ms,eth price,date_obj_y
0,1747267200,Zunami Protocol,Protocol Logic,withdrawStuckToken() exploit,500000.0,[Ethereum],False,DeFi Protocol,https://x.com/ZunamiProtocol/status/1922993510...,NaN,1201.0,Solidity,NaN,2025-05-15,2025-05-15,1747267200000,2603.718089,2025-05-15
1,1762819200,Impermax V3,Protocol Logic,Zero-Rate Hack,380000.0,[Base],False,DeFi Protocol,https://x.com/hklst4r/status/1988339762537918895,NaN,5981.0,Solidity,parent#impermax-finance,2025-11-11,2025-11-11,1762819200000,3564.610596,2025-11-11
2,1743465600,UPCX,Protocol Logic,Upgraded ProxyAdmin + withdrawByAdmin Hack,NaN,[Ethereum],False,DeFi Protocol,https://x.com/cyversalerts/status/190704694190...,NaN,NaN,Solidity,NaN,2025-04-01,2025-04-01,1743465600000,1824.207137,2025-04-01
3,1739232000,four.meme,Protocol Logic,Liquidity Pool Exploit,183000.0,[BSC],False,DeFi Protocol,https://x.com/peckshieldalert/status/188921000...,NaN,5174.0,Solidity,NaN,2025-02-11,2025-02-11,1739232000000,2660.296800,2025-02-11
4,1756166400,BetterBank,Protocol Logic,Infinite Mint and Dump,5000000.0,[PulseChain],False,DeFi Protocol,https://x.com/BetterBank_io/status/19604093916...,NaN,6611.0,Solidity,NaN,2025-08-26,2025-08-26,1756166400000,4381.626176,2025-08-26
